## Research VectorStore

### A question answering agent that is an expert knowledge worker
### To be used by employees of Rabobank
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

### Divide our documents into chunks

In [5]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from pypdf import PdfReader
from langchain_community.document_loaders.pdf import PyPDFLoader


load_dotenv(override=True)

True

In [6]:
MODEL = os.getenv("LLM_MODEL", "gpt-4o")
db_name = os.path.join(os.getenv("VECTOR_STORE_DIR"), "vector_store.db")
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-naYL4


In [7]:
def read(file_path: str) -> str:
    text = str()
    for page in PdfReader(file_path).pages:
        text += page.extract_text()
    return text

In [8]:
# How many characters in all the documents?
entire_knowledge_base = ""

documents = [
    os.path.join(os.getenv("MEMORY_DIR"), 'knowledge', "Annual Report 2025.pdf"),
    os.path.join(os.getenv("MEMORY_DIR"), 'knowledge', "Annual Report 2024.pdf"),
    os.path.join(os.getenv("MEMORY_DIR"), 'knowledge', "Annual Report 2023.pdf"),
    os.path.join(os.getenv("MEMORY_DIR"), 'knowledge', "Annual Report 2022 (EN).pdf"),
]

# read pdf file
for file_path in documents:    
    entire_knowledge_base += read(file_path)
    entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Total characters in knowledge base: 4,689,784


In [9]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4o: 1,043,018


In [11]:
# Load in everything in the knowledgebase using LangChain's loaders
documents = []
doc_type = 'Annual Report'
folder = os.path.join(os.getenv("MEMORY_DIR"), 'knowledge')
loader = DirectoryLoader(folder, glob="Annual Report*.pdf", loader_cls=PyPDFLoader)
folder_docs = loader.load()
for doc in folder_docs:
    doc.metadata["doc_type"] = doc_type
    documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 1389 documents


In [12]:
documents[1]

Document(metadata={'producer': 'Antenna House PDF Output Library 7.4.1908', 'creator': 'Antenna House XSL Formatter V7.4 MR5 Linux : 7.4.6.64781 (2024-08-22T17:49+09)', 'creationdate': '2025-03-03T16:21:27+01:00', 'title': 'Annual Report 2024', 'author': 'TIM.VAN.VILSTEREN', 'moddate': '2025-03-03T16:21:27+01:00', 'trapped': '/False', 'source': '/home/edejong/workspace/assessment-app/memory/knowledge/Annual Report 2024.pdf', 'total_pages': 375, 'page': 1, 'page_label': '2', 'doc_type': 'Annual Report'}, page_content='Preparation of the Annual Report\nThe production process of our Annual Report is as follows: the Managing Board installs an Integrated \nSustainability Reporting Group (hereafter Reporting Group). The following disciplines are represented in \nthe Reporting Group: Managing Board Secretariat, Finance, HR, Legal, Investor Relations & Rating Agencies, \nGroup Sustainability, Integrated Risk Management, Compliance and Communications and Corporate \nAffairs. The Reporting Group

In [13]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 19623 chunks
First chunk:

page_content='ANNUAL report 2024
Annual Report 2024' metadata={'producer': 'Antenna House PDF Output Library 7.4.1908', 'creator': 'Antenna House XSL Formatter V7.4 MR5 Linux : 7.4.6.64781 (2024-08-22T17:49+09)', 'creationdate': '2025-03-03T16:21:27+01:00', 'title': 'Annual Report 2024', 'author': 'TIM.VAN.VILSTEREN', 'moddate': '2025-03-03T16:21:27+01:00', 'trapped': '/False', 'source': '/home/edejong/workspace/assessment-app/memory/knowledge/Annual Report 2024.pdf', 'total_pages': 375, 'page': 0, 'page_label': '1', 'doc_type': 'Annual Report'}


In [14]:
chunks[50]

Document(metadata={'producer': 'Antenna House PDF Output Library 7.4.1908', 'creator': 'Antenna House XSL Formatter V7.4 MR5 Linux : 7.4.6.64781 (2024-08-22T17:49+09)', 'creationdate': '2025-03-03T16:21:27+01:00', 'title': 'Annual Report 2024', 'author': 'TIM.VAN.VILSTEREN', 'moddate': '2025-03-03T16:21:27+01:00', 'trapped': '/False', 'source': '/home/edejong/workspace/assessment-app/memory/knowledge/Annual Report 2024.pdf', 'total_pages': 375, 'page': 6, 'page_label': '7', 'doc_type': 'Annual Report'}, page_content='amongst others by growing demand for solutions to finance the energy transition. Performance at our vendor lease\nsubsidiary DLL remained strong thanks to higher volumes of new business.')

### Make vectors and store in Chroma

In [16]:
# Pick an embedding model

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 19623 documents


In [17]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 19,623 vectors with 3,072 dimensions in the vector store


### Visualize the data

In [18]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue'][['Annual Report'].index(t)] for t in doc_types]

In [20]:
import nbformat
print(nbformat.__version__)

5.10.4


In [21]:
import plotly.io as pio
print(pio.renderers)

Renderers configuration
-----------------------
    Default renderer: 'plotly_mimetype'
    Available renderers:
        ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode',
         'notebook', 'notebook_connected', 'kaggle', 'azure', 'colab',
         'cocalc', 'databricks', 'json', 'png', 'jpeg', 'jpg', 'svg',
         'pdf', 'browser', 'firefox', 'chrome', 'chromium', 'iframe',
         'iframe_connected', 'sphinx_gallery', 'sphinx_gallery_png']



In [22]:
pio.renderers.default = "browser"

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [23]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()